# PS S6E6 - 002 LightGBM strict raw baseline

Experiment: `exp_20260601_002_lgb_strict_raw_baseline`

Purpose:
- Build the first strict raw LightGBM baseline.
- Use only competition train/test.
- Do not use original dataset.
- Do not add feature engineering, color indices, original priors, snap features, or bias search.
- Save OOF / test prediction / CV result / submission / memo / registry.

Metric:
- `balanced_accuracy_score`

CV:
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
except Exception as e:
    raise ImportError("LightGBM import failed. On Kaggle, lightgbm should be available by default.") from e


COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260601_002_lgb_strict_raw_baseline"
SEED = 42
N_SPLITS = 5

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

print("LightGBM version:", lgb.__version__)
print("OUTDIR:", OUTDIR)

LightGBM version: 4.6.0
OUTDIR: /kaggle/working/exp_20260601_002_lgb_strict_raw_baseline


In [2]:
# ============================================================
# 1. Load data
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

print("train:", train.shape)
print("test :", test.shape)
print("sample:", sample.shape)
print("train columns:", train.columns.tolist())
print("test columns :", test.columns.tolist())
print("sample columns:", sample.columns.tolist())

display(train.head())
display(test.head())
display(sample.head())

train: (577347, 12)
test : (247435, 11)
sample: (247435, 2)
train columns: ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'class']
test columns : ['id', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population']
sample columns: ['id', 'class']


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,GALAXY
4,577351,GALAXY


In [3]:
# ============================================================
# 2. Schema fixed from 001
# ============================================================

ID_COL = "id"
TARGET_COL = "class"

NUMERIC_COLS = [
    "alpha", "delta", "u", "g", "r", "i", "z", "redshift"
]

CATEGORICAL_COLS = [
    "spectral_type",
    "galaxy_population",
]

FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS

assert ID_COL in train.columns
assert TARGET_COL in train.columns
assert ID_COL in test.columns
assert TARGET_COL not in test.columns
assert set(FEATURE_COLS).issubset(train.columns)
assert set(FEATURE_COLS).issubset(test.columns)

print("FEATURE_COLS:", FEATURE_COLS)
print("NUMERIC_COLS:", NUMERIC_COLS)
print("CATEGORICAL_COLS:", CATEGORICAL_COLS)

print("\\nTarget distribution:")
target_dist = train[TARGET_COL].value_counts().to_frame("count")
target_dist["rate"] = train[TARGET_COL].value_counts(normalize=True)
display(target_dist)

FEATURE_COLS: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population']
NUMERIC_COLS: ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
CATEGORICAL_COLS: ['spectral_type', 'galaxy_population']
\nTarget distribution:


,count,rate
class,,
GALAXY,377480,0.653818
QSO,117143,0.202899
STAR,82724,0.143283


In [4]:
# ============================================================
# 3. Strict raw preprocessing
# ============================================================

def prepare_strict_raw(train_df: pd.DataFrame, test_df: pd.DataFrame):
    X = train_df[FEATURE_COLS].copy()
    X_test = test_df[FEATURE_COLS].copy()

    # LightGBM can consume pandas categorical dtype.
    # Fit category vocabulary on train+test to avoid unseen-category dtype mismatch.
    for col in CATEGORICAL_COLS:
        all_values = pd.concat([X[col], X_test[col]], axis=0).astype("string").fillna("__NA__")
        categories = sorted(all_values.unique().tolist())
        X[col] = pd.Categorical(X[col].astype("string").fillna("__NA__"), categories=categories)
        X_test[col] = pd.Categorical(X_test[col].astype("string").fillna("__NA__"), categories=categories)

    for col in NUMERIC_COLS:
        X[col] = X[col].astype("float32")
        X_test[col] = X_test[col].astype("float32")

    return X, X_test

X, X_test = prepare_strict_raw(train, test)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

print("class_names:", class_names)
print("n_classes:", n_classes)
print("X dtypes:")
display(X.dtypes.to_frame("dtype"))

print("\\nEncoded target distribution:")
display(pd.Series(y).value_counts().sort_index().rename(index={i: c for i, c in enumerate(class_names)}).to_frame("count"))

class_names: ['GALAXY', 'QSO', 'STAR']
n_classes: 3
X dtypes:


,dtype
alpha,float32
delta,float32
u,float32
g,float32
r,float32
i,float32
z,float32
redshift,float32
spectral_type,category
galaxy_population,category


\nEncoded target distribution:


,count
GALAXY,377480
QSO,117143
STAR,82724


In [5]:
# ============================================================
# 4. Model params
# ============================================================

# Strict raw baseline params.
# Do not tune yet. Keep simple, stable, and reproducible.
LGB_PARAMS = dict(
    objective="multiclass",
    num_class=n_classes,
    boosting_type="gbdt",
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
    force_col_wise=True,
)

FIT_CALLBACKS = [
    lgb.early_stopping(stopping_rounds=150, verbose=False),
    lgb.log_evaluation(period=200),
]

print(json.dumps(LGB_PARAMS, indent=2))

{
  "objective": "multiclass",
  "num_class": 3,
  "boosting_type": "gbdt",
  "n_estimators": 5000,
  "learning_rate": 0.03,
  "num_leaves": 127,
  "max_depth": -1,
  "min_child_samples": 40,
  "subsample": 0.85,
  "subsample_freq": 1,
  "colsample_bytree": 0.85,
  "reg_alpha": 0.0,
  "reg_lambda": 1.0,
  "random_state": 42,
  "n_jobs": -1,
  "verbosity": -1,
  "force_col_wise": true
}


In [6]:
# ============================================================
# 5. CV training
# ============================================================

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_proba = np.zeros((len(train), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test), n_classes), dtype=np.float32)

fold_rows = []
feature_importance_rows = []
models = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n========== Fold {fold}/{N_SPLITS} ==========")

    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = LGBMClassifier(**LGB_PARAMS)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="multi_logloss",
        categorical_feature=CATEGORICAL_COLS,
        callbacks=FIT_CALLBACKS,
    )

    best_iter = model.best_iteration_ or LGB_PARAMS["n_estimators"]

    va_proba = model.predict_proba(X_va, num_iteration=best_iter)
    te_proba = model.predict_proba(X_test, num_iteration=best_iter)

    oof_proba[va_idx] = va_proba.astype(np.float32)
    test_proba += te_proba.astype(np.float32) / N_SPLITS

    va_pred = va_proba.argmax(axis=1)
    fold_score = balanced_accuracy_score(y_va, va_pred)

    fold_rows.append({
        "fold": fold,
        "balanced_accuracy": float(fold_score),
        "best_iteration": int(best_iter),
        "n_train": int(len(tr_idx)),
        "n_valid": int(len(va_idx)),
    })

    fi = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance_gain": model.booster_.feature_importance(importance_type="gain"),
        "importance_split": model.booster_.feature_importance(importance_type="split"),
        "fold": fold,
    })
    feature_importance_rows.append(fi)

    print("fold balanced_accuracy:", fold_score)
    print("best_iteration:", best_iter)
    print("confusion matrix:")
    print(confusion_matrix(y_va, va_pred))

    models.append(model)

fold_scores = pd.DataFrame(fold_rows)
feature_importance = pd.concat(feature_importance_rows, ignore_index=True)

oof_pred = oof_proba.argmax(axis=1)
cv_score = balanced_accuracy_score(y, oof_pred)

print("\n========== CV result ==========")
display(fold_scores)
print("OOF balanced_accuracy:", cv_score)
print("\nClassification report:")
print(classification_report(y, oof_pred, target_names=class_names))


========== Fold 1/5 ==========
[200]	valid_0's multi_logloss: 0.093272
[400]	valid_0's multi_logloss: 0.0886439
[600]	valid_0's multi_logloss: 0.0877536
[800]	valid_0's multi_logloss: 0.0874447
fold balanced_accuracy: 0.95602653803731
best_iteration: 837
confusion matrix:
[[73898   701   897]
 [  632 22600   197]
 [ 1153    94 15298]]

========== Fold 2/5 ==========
[200]	valid_0's multi_logloss: 0.0934812
[400]	valid_0's multi_logloss: 0.0891481
[600]	valid_0's multi_logloss: 0.0883926
[800]	valid_0's multi_logloss: 0.0881961
fold balanced_accuracy: 0.9579558904746607
best_iteration: 834
confusion matrix:
[[73877   644   975]
 [  591 22643   195]
 [ 1079    98 15368]]

========== Fold 3/5 ==========
[200]	valid_0's multi_logloss: 0.0950391
[400]	valid_0's multi_logloss: 0.0905049
[600]	valid_0's multi_logloss: 0.089485
[800]	valid_0's multi_logloss: 0.0894027
fold balanced_accuracy: 0.9568711594269406
best_iteration: 749
confusion matrix:
[[73789   705  1002]
 [  620 22597   212]
 [ 

,fold,balanced_accuracy,best_iteration,n_train,n_valid
0,1,0.956027,837,461877,115470
1,2,0.957956,834,461877,115470
2,3,0.956871,749,461878,115469
3,4,0.956645,837,461878,115469
4,5,0.957033,730,461878,115469


OOF balanced_accuracy: 0.9569063401002502

Classification report:
              precision    recall  f1-score   support

      GALAXY       0.98      0.98      0.98    377480
         QSO       0.97      0.97      0.97    117143
        STAR       0.93      0.93      0.93     82724

    accuracy                           0.97    577347
   macro avg       0.96      0.96      0.96    577347
weighted avg       0.97      0.97      0.97    577347



In [7]:
# ============================================================
# 6. Submission
# ============================================================

test_pred = test_proba.argmax(axis=1)
test_labels = le.inverse_transform(test_pred)

submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: test_labels,
})

# Validate submission schema
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

print(submission[TARGET_COL].value_counts(normalize=True))
display(submission.head())

submission_path = OUTDIR / "submission_lgb_strict_raw_baseline.csv"
submission.to_csv(submission_path, index=False)
print("saved:", submission_path)

class
GALAXY    0.654984
QSO       0.202263
STAR      0.142753
Name: proportion, dtype: float64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


saved: /kaggle/working/exp_20260601_002_lgb_strict_raw_baseline/submission_lgb_strict_raw_baseline.csv


In [8]:
# ============================================================
# 7. Save artifacts
# ============================================================

np.save(OUTDIR / "oof_lgb_strict_raw_proba.npy", oof_proba)
np.save(OUTDIR / "pred_lgb_strict_raw_proba.npy", test_proba)

oof_df = pd.DataFrame({
    ID_COL: train[ID_COL].values,
    "y_true": train[TARGET_COL].astype(str).values,
    "y_pred": le.inverse_transform(oof_pred),
})
for i, cls in enumerate(class_names):
    oof_df[f"proba_{cls}"] = oof_proba[:, i]
oof_df.to_csv(OUTDIR / "oof_lgb_strict_raw.csv", index=False)

pred_df = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "pred_label": test_labels,
})
for i, cls in enumerate(class_names):
    pred_df[f"proba_{cls}"] = test_proba[:, i]
pred_df.to_csv(OUTDIR / "pred_lgb_strict_raw.csv", index=False)

fold_scores.to_csv(OUTDIR / "fold_scores.csv", index=False)

feature_importance_summary = (
    feature_importance
    .groupby("feature", as_index=False)[["importance_gain", "importance_split"]]
    .mean()
    .sort_values("importance_gain", ascending=False)
)
feature_importance.to_csv(OUTDIR / "feature_importance_by_fold.csv", index=False)
feature_importance_summary.to_csv(OUTDIR / "feature_importance.csv", index=False)

cv_result = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "model": "LightGBM",
    "status": "completed",
    "metric": "balanced_accuracy_score",
    "cv_score": float(cv_score),
    "fold_scores": fold_scores.to_dict(orient="records"),
    "class_names": class_names,
    "label_mapping": {str(cls): int(i) for i, cls in enumerate(class_names)},
    "feature_cols": FEATURE_COLS,
    "numeric_cols": NUMERIC_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "params": LGB_PARAMS,
    "use_original": False,
    "use_fe": False,
    "use_bias_search": False,
    "submission_path": str(submission_path),
}

with open(OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(cv_result, f, indent=2, ensure_ascii=False, default=str)

print("Artifacts saved to:", OUTDIR)
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Artifacts saved to: /kaggle/working/exp_20260601_002_lgb_strict_raw_baseline
 - cv_result.json
 - feature_importance.csv
 - feature_importance_by_fold.csv
 - fold_scores.csv
 - oof_lgb_strict_raw.csv
 - oof_lgb_strict_raw_proba.npy
 - pred_lgb_strict_raw.csv
 - pred_lgb_strict_raw_proba.npy
 - submission_lgb_strict_raw_baseline.csv


In [9]:
# ============================================================
# 8. Initialize / update oof_registry.csv
# ============================================================

registry_path = WORKDIR / "oof_registry.csv"

registry_row = {
    "exp_id": EXP_ID,
    "model_family": "LightGBM",
    "feature_family": "strict_raw",
    "cv_metric": "balanced_accuracy_score",
    "cv_score": float(cv_score),
    "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    "public_lb": np.nan,
    "use_original": False,
    "use_fe": False,
    "use_bias_search": False,
    "oof_path": str(OUTDIR / "oof_lgb_strict_raw_proba.npy"),
    "pred_path": str(OUTDIR / "pred_lgb_strict_raw_proba.npy"),
    "submission_path": str(submission_path),
    "role": "strict_raw_baseline",
    "status": "completed",
    "keep_hold_drop": "keep_reference",
    "notes": "First LightGBM strict raw baseline. No original, no FE, no bias search.",
}

if registry_path.exists():
    registry = pd.read_csv(registry_path)
    registry = registry[registry["exp_id"] != EXP_ID]
    registry = pd.concat([registry, pd.DataFrame([registry_row])], ignore_index=True)
else:
    registry = pd.DataFrame([registry_row])

registry.to_csv(registry_path, index=False)
registry.to_csv(OUTDIR / "oof_registry.csv", index=False)

display(registry.tail())
print("registry saved:", registry_path)

,exp_id,model_family,feature_family,cv_metric,cv_score,fold_std,public_lb,use_original,use_fe,use_bias_search,oof_path,pred_path,submission_path,role,status,keep_hold_drop,notes
0,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,strict_raw,balanced_accuracy_score,0.956906,0.000626,NaN,False,False,False,/kaggle/working/exp_20260601_002_lgb_strict_ra...,/kaggle/working/exp_20260601_002_lgb_strict_ra...,/kaggle/working/exp_20260601_002_lgb_strict_ra...,strict_raw_baseline,completed,keep_reference,First LightGBM strict raw baseline. No origina...


registry saved: /kaggle/working/oof_registry.csv


In [10]:
# ============================================================
# 9. memo.yml
# ============================================================

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "LightGBM strict raw baseline",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "model": "LightGBM",
    },
    "objective": {
        "summary": (
            "Build the first strict raw LightGBM baseline using only competition train/test. "
            "No original dataset, no feature engineering, no color indices, no bias search."
        ),
        "success_criteria": [
            "save oof proba",
            "save test pred proba",
            "save cv_result",
            "save submission",
            "save feature importance",
            "update oof_registry",
        ],
    },
    "data": {
        "train_path": str(TRAIN_PATH),
        "test_path": str(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUB_PATH),
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "target_col": TARGET_COL,
        "id_col": ID_COL,
        "use_original": False,
    },
    "features": {
        "feature_family": "strict_raw",
        "feature_cols": FEATURE_COLS,
        "numeric_cols": NUMERIC_COLS,
        "categorical_cols": CATEGORICAL_COLS,
        "notes": "Raw competition columns only.",
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "score": float(cv_score),
        "fold_scores": fold_scores.to_dict(orient="records"),
        "fold_std": float(fold_scores["balanced_accuracy"].std(ddof=0)),
    },
    "params": LGB_PARAMS,
    "outputs": {
        "oof_proba": "oof_lgb_strict_raw_proba.npy",
        "pred_proba": "pred_lgb_strict_raw_proba.npy",
        "oof_csv": "oof_lgb_strict_raw.csv",
        "pred_csv": "pred_lgb_strict_raw.csv",
        "submission": "submission_lgb_strict_raw_baseline.csv",
        "cv_result": "cv_result.json",
        "feature_importance": "feature_importance.csv",
        "registry": "oof_registry.csv",
    },
    "judgement": {
        "status": "keep_reference",
        "reason": (
            "This is the first strict raw LightGBM baseline and should be kept as the reference "
            "for CatBoost/XGBoost baselines and later FE/original/bias-search experiments."
        ),
        "next_action": [
            "Run exp_20260601_003_cat_strict_raw_baseline",
            "Run exp_20260601_004_xgb_strict_raw_baseline",
            "Compare OOF correlation and fold stability after 002/003/004",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("memo saved:", memo_path)

memo saved: /kaggle/working/exp_20260601_002_lgb_strict_raw_baseline/memo.yml
